# 11 — Schema Drift Guard Pattern

Detect and handle schema changes in incoming data.

Process:

INPUT DF
  |
  v
COMPARE WITH EXPECTED SCHEMA
  |
  +--> OK
  +--> EXTRA COLUMNS
  +--> MISSING COLUMNS

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("schema-drift-guard")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## Step 1 — Input

In [2]:
df = spark.createDataFrame(
    [("e1","u1",10.0,"extra")],
    ["event_id","user_id","amount","new_col"]
)

df.show()

+--------+-------+------+-------+
|event_id|user_id|amount|new_col|
+--------+-------+------+-------+
|      e1|     u1|  10.0|  extra|
+--------+-------+------+-------+



## Step 2 — Expected schema

In [3]:
expected_columns = {"event_id","user_id","amount"}

## Step 3 — Detect drift

In [4]:
incoming = set(df.columns)

extra = incoming - expected_columns
missing = expected_columns - incoming

print("extra:", extra)
print("missing:", missing)

extra: {'new_col'}
missing: set()


## Step 4 — Handle drift

In [5]:
df_clean = df

if extra:
    df_clean = df_clean.drop(*extra)

for col in missing:
    df_clean = df_clean.withColumn(col, F.lit(None))

df_clean = df_clean.select(*expected_columns)
df_clean.show()

+------+--------+-------+
|amount|event_id|user_id|
+------+--------+-------+
|  10.0|      e1|     u1|
+------+--------+-------+



## Step 5 — Control

In [6]:
print("final columns:", df_clean.columns)

final columns: ['amount', 'event_id', 'user_id']


In [7]:
spark.stop()